# Fix Verification — Policy Regression Tests

Tests the four bugs fixed:
1. `associated_symptoms` (and other list targets) looping forever
2. LLM JSON failures on relieving_factors / list fields → moved to rule-based
3. Back pain triggering `end_intake_early` via `'worse'` matching escalation signals
4. Chest pain + SOB flag not firing because SOB went to `policy_answers` not `associated_symptoms`

**Run the reload cell first after copying the fixed files into place.**

In [1]:
from IPython.display import display
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
import pandas as pd
pd.set_option('display.max_rows', None)

In [3]:
import importlib
import intake_engine.policies.complaint_policies
importlib.reload(intake_engine.policies.complaint_policies)

import intake_engine.policies.target_specs
import intake_engine.policies.base_complaint_policy
import intake_engine.policies.complaint_policies
import intake_engine.policies.policy_selector
import intake_engine.state.templates
import intake_engine.state.routing
import intake_engine.state.rule_based_parser
import intake_engine.llm_extractor
import intake_engine.IntakeState
import intake_engine.app_flow

importlib.reload(intake_engine.policies.target_specs)
importlib.reload(intake_engine.policies.base_complaint_policy)
importlib.reload(intake_engine.policies.complaint_policies)
importlib.reload(intake_engine.policies.policy_selector)
importlib.reload(intake_engine.state.templates)
importlib.reload(intake_engine.state.routing)
importlib.reload(intake_engine.state.rule_based_parser)
importlib.reload(intake_engine.llm_extractor)
importlib.reload(intake_engine.IntakeState)
importlib.reload(intake_engine.app_flow)

print("Reloaded.")

<module 'intake_engine.policies.complaint_policies' from 'c:\\Source\\689\\Capstone2026_Spring\\intake_engine\\policies\\complaint_policies.py'>

<module 'intake_engine.policies.target_specs' from 'c:\\Source\\689\\Capstone2026_Spring\\intake_engine\\policies\\target_specs.py'>

<module 'intake_engine.policies.base_complaint_policy' from 'c:\\Source\\689\\Capstone2026_Spring\\intake_engine\\policies\\base_complaint_policy.py'>

<module 'intake_engine.policies.complaint_policies' from 'c:\\Source\\689\\Capstone2026_Spring\\intake_engine\\policies\\complaint_policies.py'>

<module 'intake_engine.policies.policy_selector' from 'c:\\Source\\689\\Capstone2026_Spring\\intake_engine\\policies\\policy_selector.py'>

...

<module 'intake_engine.state.rule_based_parser' from 'c:\\Source\\689\\Capstone2026_Spring\\intake_engine\\state\\rule_based_parser.py'>

<module 'intake_engine.llm_extractor' from 'c:\\Source\\689\\Capstone2026_Spring\\intake_engine\\llm_extractor.py'>

<module 'intake_engine.IntakeState' from 'c:\\Source\\689\\Capstone2026_Spring\\intake_engine\\IntakeState.py'>

<module 'intake_engine.app_flow' from 'c:\\Source\\689\\Capstone2026_Spring\\intake_engine\\app_flow.py'>

Reloaded.


In [4]:
from pprint import pprint
from intake_engine.app_flow import IntakeAppFlow


def run_policy_test(session_name, primary_complaint, answers, label=None):
    label = label or session_name
    print("=" * 70)
    print(f"TEST: {label}")
    print("=" * 70)

    app = IntakeAppFlow(conn=None)
    app.new_session(
        session_name=session_name,
        primary_complaint=primary_complaint,
        auto_save=False,
    )

    first = app.start_intake(auto_save=False)
    intents = [first["intent"]]

    for answer in answers:
        result = app.submit_answer(answer, auto_save=False)
        intents.append(result["next_intent"])

    state = app.get_state()

    print("\nINTENT SEQUENCE:")
    for i, intent in enumerate(intents):
        print(f"  {i:>2}. {intent}")

    print("\nFINAL HPI:")
    pprint(state["hpi"])

    print("\nFINAL POLICY ANSWERS (non-None only):")
    pprint({k: v for k, v in state["policy_answers"].items() if v is not None})

    print("\nQUESTION STATUS:")
    pprint(state["conversation_meta"]["question_status"])

    print("\nMISSING CLARIFICATIONS:")
    pprint(state["missing_clarifications"])

    print("\nFLAGS:", state["flags"])
    print("INTAKE COMPLETE:", state["conversation_meta"]["intake_complete"])
    print()
    return app


print("Helper loaded.")

Helper loaded.


## Fix 1 — List targets resolve after one answer
Previously `associated_symptoms`, `medications`, `allergies` etc. would loop forever
because `asked_answered` on a list target didn't count as resolved.

**Expected:** after answering `associated_symptoms`, routing moves on. No repeated intents.

In [5]:
from pprint import pprint
from intake_engine.app_flow import IntakeAppFlow

app = IntakeAppFlow(conn=None)
app.new_session(session_name="list_resolve_test", primary_complaint="headache", auto_save=False)
app.start_intake(auto_save=False)

# Run through to associated_symptoms
for answer in [
    "I have a headache.",
    "No.",                                                      # sudden_severe_onset
    "No weakness, numbness, trouble speaking, or similar.",     # neurologic_symptoms
    "No vision changes.",                                       # visual_changes
    "No confusion.",                                            # confusion_or_ams
    "No fever or neck stiffness.",                              # fever_or_neck_stiffness
    "No recent head trauma.",                                   # head_trauma
    "No, I am not pregnant.",                                   # pregnancy_or_postpartum_context
    "This morning.",                                            # onset
    "About four hours.",                                        # duration
    "7 out of 10.",                                             # severity
    "It comes and goes.",                                       # timing
    "It has stayed about the same.",                            # course
    "In the front of my head.",                                 # location
    "Throbbing.",                                               # character
    "Bright light makes it worse.",                             # aggravating_factors
    "Rest helps a little.",                                     # relieving_factors
]:
    app.submit_answer(answer, auto_save=False)

# Now answer associated_symptoms — should move on, not loop
r = app.submit_answer("Nausea and sensitivity to light.", auto_save=False)
print("After associated_symptoms answer:")
print("  current_intent:", r["current_intent"])
print("  next_intent:", r["next_intent"])  # Should NOT be ask_associated_symptoms
print("  associated_symptoms in HPI:", app.get_state()["hpi"]["associated_symptoms"])
print("  question_status[associated_symptoms]:",
      app.get_state()["conversation_meta"]["question_status"].get("associated_symptoms"))
print()

assert r["next_intent"] != "ask_associated_symptoms", "FAIL: still looping on associated_symptoms"
print("PASS: associated_symptoms resolved after one answer")

{'chief_complaint_primary': None,
 'other_concerns': [],
 'hpi': {'summary': None,
  'onset': None,
  'location': None,
  'duration': None,
  'timing': None,
  'course': None,
  'character': None,
  'severity': None,
  'aggravating_factors': [],
  'relieving_factors': [],
  'associated_symptoms': [],
  'context': None},
 'pertinent_positives': [],
 'pertinent_negatives': [],
 'pmh_psh': {'past_medical_history': [], 'past_surgical_history': []},
 'medications': [],
 'allergies': [],
 'social_factors': [],
 'missing_clarifications': [],
 'flags': [],
 'policy_answers': {'sudden_severe_onset': None,
  'neurologic_symptoms': None,
  'neurologic_symptom_terms': [],
  'visual_changes': None,
  'confusion_or_ams': None,
  'fever_or_neck_stiffness': None,
  'head_trauma': None,
  'pregnancy_or_postpartum_context': None,
  'photophobia': None,
  'phonophobia': None,
  'nausea_or_vomiting': None,
  'aura_features': [],
  'exertional_trigger': None,
  'positional_component': None,
  'new_or_progr

{'intent': 'ask_main_reason_for_visit',
 'target': None,
 'question': "Could you tell me what's been bothering you today?"}

{'current_intent': 'ask_main_reason_for_visit',
 'applied_update': {'set_fields': {'chief_complaint_primary': 'headache',
   'conversation_meta.last_answer_status': 'answered',
   'conversation_meta.early_exit_reason': None},
  'append_fields': {},
  'flags_to_add': [],
  'missing_clarifications_to_add': []},
 'missing_clarifications': ['when it started',
  'where the symptom is located',
  'how long it has been going on',
  'how severe it is',
  'associated symptoms'],
 'next_intent': 'ask_sudden_severe_onset',
 'next_target': 'sudden_severe_onset',
 'next_question': 'Did the headache start suddenly and become very severe right away?'}

{'current_intent': 'ask_sudden_severe_onset',
 'applied_update': {'set_fields': {'conversation_meta.last_answer_status': 'answered',
   'conversation_meta.early_exit_reason': None,
   'policy_answers.sudden_severe_onset': False},
  'append_fields': {},
  'flags_to_add': [],
  'missing_clarifications_to_add': []},
 'missing_clarifications': ['when it started',
  'where the symptom is located',
  'how long it has been going on',
  'how severe it is',
  'associated symptoms'],
 'next_intent': 'ask_neurologic_symptoms',
 'next_target': 'neurologic_symptoms',
 'next_question': 'Have you had weakness, numbness, trouble speaking, confusion, or vision changes?'}

{'current_intent': 'ask_neurologic_symptoms',
 'applied_update': {'set_fields': {'policy_answers.neurologic_symptoms': False,
   'policy_answers.neurologic_symptom_terms': [],
   'conversation_meta.last_answer_status': 'answered',
   'conversation_meta.early_exit_reason': None},
  'append_fields': {},
  'flags_to_add': [],
  'missing_clarifications_to_add': []},
 'missing_clarifications': ['when it started',
  'where the symptom is located',
  'how long it has been going on',
  'how severe it is',
  'associated symptoms'],
 'next_intent': 'ask_visual_changes',
 'next_target': 'visual_changes',
 'next_question': 'Have you had blurry vision, double vision, or other vision changes?'}

...

{'current_intent': 'ask_location',
 'applied_update': {'set_fields': {'conversation_meta.last_answer_status': 'answered',
   'conversation_meta.early_exit_reason': None,
   'hpi.location': 'In the front of my head'},
  'append_fields': {},
  'flags_to_add': [],
  'missing_clarifications_to_add': []},
 'missing_clarifications': ['associated symptoms'],
 'next_intent': 'ask_character',
 'next_target': 'character',
 'next_question': 'Can you describe what the headache feels like?'}

{'current_intent': 'ask_character',
 'applied_update': {'set_fields': {'conversation_meta.last_answer_status': 'answered',
   'conversation_meta.early_exit_reason': None,
   'hpi.character': 'Throbbing'},
  'append_fields': {},
  'flags_to_add': [],
  'missing_clarifications_to_add': []},
 'missing_clarifications': ['associated symptoms'],
 'next_intent': 'ask_aggravating_factors',
 'next_target': 'aggravating_factors',
 'next_question': 'Can you tell me what makes your headache worse?'}

{'current_intent': 'ask_aggravating_factors',
 'applied_update': {'set_fields': {'conversation_meta.last_answer_status': 'answered',
   'conversation_meta.early_exit_reason': None},
  'append_fields': {'hpi.aggravating_factors': ['bright light makes it worse']},
  'flags_to_add': [],
  'missing_clarifications_to_add': []},
 'missing_clarifications': ['associated symptoms'],
 'next_intent': 'ask_relieving_factors',
 'next_target': 'relieving_factors',
 'next_question': 'What makes the headache feel better?'}

{'current_intent': 'ask_relieving_factors',
 'applied_update': {'set_fields': {'conversation_meta.last_answer_status': 'answered',
   'conversation_meta.early_exit_reason': None},
  'append_fields': {'hpi.relieving_factors': ['rest helps a little']},
  'flags_to_add': [],
  'missing_clarifications_to_add': []},
 'missing_clarifications': ['associated symptoms'],
 'next_intent': 'ask_associated_symptoms',
 'next_target': 'associated_symptoms',
 'next_question': 'Are there any other symptoms you’re experiencing with the headache?'}

After associated_symptoms answer:
  current_intent: ask_associated_symptoms
  next_intent: ask_medications
  associated_symptoms in HPI: ['nausea', 'sensitivity to light']
  question_status[associated_symptoms]: asked_answered

PASS: associated_symptoms resolved after one answer


## Fix 2 — List fields now rule-based, no LLM JSON errors
**Expected:** no EXTRACTOR FALLBACK errors, aggravating/relieving factors parse cleanly, no trailing punctuation.

In [6]:
from intake_engine.app_flow import IntakeAppFlow

app = IntakeAppFlow(conn=None)
app.new_session(session_name="list_parse_test", primary_complaint="headache", auto_save=False)
app.start_intake(auto_save=False)

for answer in [
    "I have a headache.",
    "No.", "No weakness.", "No vision changes.", "No confusion.",
    "No fever.", "No head trauma.", "No, not pregnant.",
    "This morning.", "About four hours.", "7 out of 10.",
    "It comes and goes.", "It has stayed about the same.",
    "In the front of my head.", "Throbbing.",
]:
    app.submit_answer(answer, auto_save=False)

r_agg = app.submit_answer("Bright light makes it worse.", auto_save=False)
r_rel = app.submit_answer("Rest helps a little.", auto_save=False)
r_sym = app.submit_answer("Nausea and vomiting.", auto_save=False)

state = app.get_state()
print("aggravating_factors:", state["hpi"]["aggravating_factors"])
print("relieving_factors:", state["hpi"]["relieving_factors"])
print("associated_symptoms:", state["hpi"]["associated_symptoms"])
print()

# Check no trailing punctuation
for field, values in [
    ("aggravating_factors", state["hpi"]["aggravating_factors"]),
    ("relieving_factors", state["hpi"]["relieving_factors"]),
    ("associated_symptoms", state["hpi"]["associated_symptoms"]),
]:
    for v in values:
        assert not v.endswith("."), f"FAIL: trailing period in {field}: '{v}'"

print("PASS: no trailing punctuation in list fields")

{'chief_complaint_primary': None,
 'other_concerns': [],
 'hpi': {'summary': None,
  'onset': None,
  'location': None,
  'duration': None,
  'timing': None,
  'course': None,
  'character': None,
  'severity': None,
  'aggravating_factors': [],
  'relieving_factors': [],
  'associated_symptoms': [],
  'context': None},
 'pertinent_positives': [],
 'pertinent_negatives': [],
 'pmh_psh': {'past_medical_history': [], 'past_surgical_history': []},
 'medications': [],
 'allergies': [],
 'social_factors': [],
 'missing_clarifications': [],
 'flags': [],
 'policy_answers': {'sudden_severe_onset': None,
  'neurologic_symptoms': None,
  'neurologic_symptom_terms': [],
  'visual_changes': None,
  'confusion_or_ams': None,
  'fever_or_neck_stiffness': None,
  'head_trauma': None,
  'pregnancy_or_postpartum_context': None,
  'photophobia': None,
  'phonophobia': None,
  'nausea_or_vomiting': None,
  'aura_features': [],
  'exertional_trigger': None,
  'positional_component': None,
  'new_or_progr

{'intent': 'ask_main_reason_for_visit',
 'target': None,
 'question': "Could you tell me what's been bothering you today?"}

{'current_intent': 'ask_main_reason_for_visit',
 'applied_update': {'set_fields': {'chief_complaint_primary': 'headache',
   'conversation_meta.last_answer_status': 'answered',
   'conversation_meta.early_exit_reason': None},
  'append_fields': {},
  'flags_to_add': [],
  'missing_clarifications_to_add': []},
 'missing_clarifications': ['when it started',
  'where the symptom is located',
  'how long it has been going on',
  'how severe it is',
  'associated symptoms'],
 'next_intent': 'ask_sudden_severe_onset',
 'next_target': 'sudden_severe_onset',
 'next_question': 'Did the headache start suddenly and become very severe right away?'}

{'current_intent': 'ask_sudden_severe_onset',
 'applied_update': {'set_fields': {'conversation_meta.last_answer_status': 'answered',
   'conversation_meta.early_exit_reason': None,
   'policy_answers.sudden_severe_onset': False},
  'append_fields': {},
  'flags_to_add': [],
  'missing_clarifications_to_add': []},
 'missing_clarifications': ['when it started',
  'where the symptom is located',
  'how long it has been going on',
  'how severe it is',
  'associated symptoms'],
 'next_intent': 'ask_neurologic_symptoms',
 'next_target': 'neurologic_symptoms',
 'next_question': 'Have you had weakness, numbness, trouble speaking, confusion, or vision changes?'}

{'current_intent': 'ask_neurologic_symptoms',
 'applied_update': {'set_fields': {'policy_answers.neurologic_symptoms': False,
   'policy_answers.neurologic_symptom_terms': [],
   'conversation_meta.last_answer_status': 'answered',
   'conversation_meta.early_exit_reason': None},
  'append_fields': {},
  'flags_to_add': [],
  'missing_clarifications_to_add': []},
 'missing_clarifications': ['when it started',
  'where the symptom is located',
  'how long it has been going on',
  'how severe it is',
  'associated symptoms'],
 'next_intent': 'ask_visual_changes',
 'next_target': 'visual_changes',
 'next_question': 'Have you had blurry vision, double vision, or other vision changes?'}

...

{'current_intent': 'ask_timing',
 'applied_update': {'set_fields': {'conversation_meta.last_answer_status': 'answered',
   'conversation_meta.early_exit_reason': None,
   'hpi.timing': 'It comes and goes'},
  'append_fields': {},
  'flags_to_add': [],
  'missing_clarifications_to_add': []},
 'missing_clarifications': ['where the symptom is located',
  'associated symptoms'],
 'next_intent': 'ask_course',
 'next_target': 'course',
 'next_question': 'Can you tell me how this headache has been changing over the last few hours?'}

{'current_intent': 'ask_course',
 'applied_update': {'set_fields': {'conversation_meta.last_answer_status': 'answered',
   'conversation_meta.early_exit_reason': None,
   'hpi.course': 'It has stayed about the same'},
  'append_fields': {},
  'flags_to_add': [],
  'missing_clarifications_to_add': []},
 'missing_clarifications': ['where the symptom is located',
  'associated symptoms'],
 'next_intent': 'ask_location',
 'next_target': 'location',
 'next_question': 'Where exactly are you feeling the headache?'}

{'current_intent': 'ask_location',
 'applied_update': {'set_fields': {'conversation_meta.last_answer_status': 'answered',
   'conversation_meta.early_exit_reason': None,
   'hpi.location': 'In the front of my head'},
  'append_fields': {},
  'flags_to_add': [],
  'missing_clarifications_to_add': []},
 'missing_clarifications': ['associated symptoms'],
 'next_intent': 'ask_character',
 'next_target': 'character',
 'next_question': 'Can you describe what the headache feels like?'}

{'current_intent': 'ask_character',
 'applied_update': {'set_fields': {'conversation_meta.last_answer_status': 'answered',
   'conversation_meta.early_exit_reason': None,
   'hpi.character': 'Throbbing'},
  'append_fields': {},
  'flags_to_add': [],
  'missing_clarifications_to_add': []},
 'missing_clarifications': ['associated symptoms'],
 'next_intent': 'ask_aggravating_factors',
 'next_target': 'aggravating_factors',
 'next_question': 'Can you tell me what makes your headache worse?'}

aggravating_factors: ['bright light makes it worse']
relieving_factors: ['rest helps a little']
associated_symptoms: ['nausea', 'vomiting']

PASS: no trailing punctuation in list fields


## Fix 3 — Back pain aggravating factors no longer trigger `end_intake_early`
Previously `'worse'` in `'Bending makes it worse.'` matched `extract_escalation_signals`
worsening terms, adding a `rapid_worsening` acuity signal.

**Expected:** back pain completes normally, intake_complete=False until wrap-up.

In [7]:
run_policy_test(
    session_name="back_pain_fix",
    primary_complaint="back pain",
    label="BACK PAIN — fix3 aggravating factors escalation",
    answers=[
        "I have low back pain.",
        "No weakness or numbness in my legs.",        # neurologic_symptoms
        "No bowel or bladder changes.",               # bowel_or_bladder_changes
        "No fever.",                                  # fever
        "No recent head trauma.",                     # head_trauma
        "No, it is not getting worse quickly.",       # rapid_worsening
        "This morning.",                              # onset
        "About six hours.",                           # duration
        "6 out of 10.",                               # severity
        "It is constant.",                            # timing
        "It has been about the same.",                # course
        "Lower back, right side.",                    # location
        "Dull and achy.",                             # character
        "Bending makes it worse.",                    # aggravating_factors — previously triggered end_intake_early
        "Lying down helps a little.",                 # relieving_factors
        "No other symptoms.",                         # associated_symptoms
        "Ibuprofen as needed.",                       # medications
        "No allergies.",                              # allergies
        "No, it does not go down my leg.",            # radiation
        "Yes, I lifted something heavy yesterday.",   # recent_heavy_lifting
        "No numbness or tingling.",                   # numbness_or_tingling
        "Nothing else.",
    ],
)

TEST: BACK PAIN — fix3 aggravating factors escalation

INTENT SEQUENCE:
   0. ask_main_reason_for_visit
   1. ask_neurologic_symptoms
   2. ask_bowel_or_bladder_changes
   3. ask_fever
   4. ask_head_trauma
   5. ask_rapid_worsening
   6. ask_onset
   7. ask_duration
   8. ask_severity
   9. ask_timing
  10. ask_course
  11. ask_location
  12. ask_character
  13. ask_aggravating_factors
  14. ask_relieving_factors
  15. ask_associated_symptoms
  16. ask_medications
  17. ask_allergies
  18. ask_radiation
  19. ask_recent_heavy_lifting
  20. ask_numbness_or_tingling
  21. summarize_and_check_for_anything_else
  22. summarize_and_check_for_anything_else

FINAL HPI:
{'aggravating_factors': ['bending makes it worse'],
 'associated_symptoms': [],
 'character': 'Dull and achy',
 'context': None,
 'course': 'It has been about the same',
 'duration': 'about six hours',
 'location': 'Lower back, right side',
 'onset': 'This morning',
 'relieving_factors': ['lying down helps a little'],
 'severi

## Fix 4 — Chest pain + SOB escalation flag fires correctly
Previously `detect_basic_flags` only checked `hpi.associated_symptoms` for SOB.
Now it also checks `policy_answers.shortness_of_breath == True`.

**Expected:** after answering `shortness_of_breath` = Yes, flag `chest_pain_with_shortness_of_breath` fires.

In [8]:
from pprint import pprint
from intake_engine.app_flow import IntakeAppFlow

app = IntakeAppFlow(conn=None)
app.new_session(session_name="escalation_test", primary_complaint="chest pain", auto_save=False)
app.start_intake(auto_save=False)

r1 = app.submit_answer("I have chest pain.", auto_save=False)
print("After chief complaint:")
print("  next_intent:", r1["next_intent"])  # should be ask_shortness_of_breath

r2 = app.submit_answer("Yes, I have shortness of breath.", auto_save=False)
print("After SOB = Yes:")
print("  next_intent:", r2["next_intent"])  # should escalate

state = app.get_state()
print("  flags:", state["flags"])
print("  policy_answers.shortness_of_breath:", state["policy_answers"]["shortness_of_breath"])
print()

assert "chest_pain_with_shortness_of_breath" in state["flags"], \
    "FAIL: chest_pain_with_shortness_of_breath flag not set"
assert r2["next_intent"] in {"escalate_high_risk_chest_pain", "recommend_immediate_attention"}, \
    f"FAIL: expected escalation intent, got {r2['next_intent']}"
print("PASS: escalation flag fires correctly")

{'chief_complaint_primary': None,
 'other_concerns': [],
 'hpi': {'summary': None,
  'onset': None,
  'location': None,
  'duration': None,
  'timing': None,
  'course': None,
  'character': None,
  'severity': None,
  'aggravating_factors': [],
  'relieving_factors': [],
  'associated_symptoms': [],
  'context': None},
 'pertinent_positives': [],
 'pertinent_negatives': [],
 'pmh_psh': {'past_medical_history': [], 'past_surgical_history': []},
 'medications': [],
 'allergies': [],
 'social_factors': [],
 'missing_clarifications': [],
 'flags': [],
 'policy_answers': {'sudden_severe_onset': None,
  'neurologic_symptoms': None,
  'neurologic_symptom_terms': [],
  'visual_changes': None,
  'confusion_or_ams': None,
  'fever_or_neck_stiffness': None,
  'head_trauma': None,
  'pregnancy_or_postpartum_context': None,
  'photophobia': None,
  'phonophobia': None,
  'nausea_or_vomiting': None,
  'aura_features': [],
  'exertional_trigger': None,
  'positional_component': None,
  'new_or_progr

{'intent': 'ask_main_reason_for_visit',
 'target': None,
 'question': "Could you tell me what's been bothering you today?"}

After chief complaint:
  next_intent: ask_shortness_of_breath
After SOB = Yes:
  next_intent: escalate_high_risk_chest_pain
  flags: ['chest_pain_with_shortness_of_breath']
  policy_answers.shortness_of_breath: True

PASS: escalation flag fires correctly


## Fix — Trailing punctuation stripped from HPI text fields

In [9]:
from intake_engine.state.rule_based_parser import extract_duration, split_free_text_list

cases = [
    (extract_duration, "About four hours.", "about four hours"),
    (extract_duration, "For two days.", "two days"),
    (split_free_text_list, "Nausea and vomiting.", ["nausea", "vomiting"]),
    (split_free_text_list, "Bright light makes it worse.", ["bright light makes it worse"]),
]

for fn, input_val, expected in cases:
    result = fn(input_val)
    assert result == expected, f"FAIL: {fn.__name__}({input_val!r}) = {result!r}, expected {expected!r}"
    print(f"PASS: {fn.__name__}({input_val!r}) -> {result!r}")

PASS: extract_duration('About four hours.') -> 'about four hours'
PASS: extract_duration('For two days.') -> 'two days'
PASS: split_free_text_list('Nausea and vomiting.') -> ['nausea', 'vomiting']
PASS: split_free_text_list('Bright light makes it worse.') -> ['bright light makes it worse']


## Full run — all seven policies

In [10]:
run_policy_test(
    session_name="headache_full",
    primary_complaint="headache",
    label="HEADACHE — full",
    answers=[
        "I have a headache.",
        "No.",
        "No weakness, numbness, trouble speaking, or similar.",
        "No vision changes.",
        "No confusion.",
        "No fever or neck stiffness.",
        "No recent head trauma.",
        "No, I am not pregnant and not postpartum.",
        "This morning.",
        "About four hours.",
        "7 out of 10.",
        "It comes and goes.",
        "It has stayed about the same.",
        "In the front of my head.",
        "Throbbing.",
        "Bright light makes it worse.",
        "Rest helps a little.",
        "Nausea and sensitivity to light.",
        "Ibuprofen 400mg as needed.",
        "No known allergies.",
        "Yes, light sensitivity.",
        "No.",
        "Yes, nausea.",
        "No aura before this headache.",
        "No, no exertion triggered it.",
        "No, position does not change it.",
        "No, this is my usual headache pattern.",
        "No, I have not been overusing pain medicine.",
        "Nothing else.",
    ],
)

TEST: HEADACHE — full

INTENT SEQUENCE:
   0. ask_main_reason_for_visit
   1. ask_sudden_severe_onset
   2. ask_neurologic_symptoms
   3. ask_visual_changes
   4. ask_confusion_or_ams
   5. ask_fever_or_neck_stiffness
   6. ask_head_trauma
   7. ask_pregnancy_or_postpartum_context
   8. ask_onset
   9. ask_duration
  10. ask_severity
  11. ask_timing
  12. ask_course
  13. ask_location
  14. ask_character
  15. ask_aggravating_factors
  16. ask_relieving_factors
  17. ask_associated_symptoms
  18. ask_medications
  19. ask_allergies
  20. ask_photophobia
  21. ask_phonophobia
  22. ask_nausea_or_vomiting
  23. ask_aura_features
  24. ask_exertional_trigger
  25. ask_positional_component
  26. ask_new_or_progressive_pattern
  27. ask_medication_overuse_context
  28. summarize_and_check_for_anything_else
  29. summarize_and_check_for_anything_else

FINAL HPI:
{'aggravating_factors': ['bright light makes it worse'],
 'associated_symptoms': ['nausea', 'sensitivity to light'],
 'character':

In [11]:
run_policy_test(
    session_name="chest_pain_full",
    primary_complaint="chest pain",
    label="CHEST PAIN — full benign",
    answers=[
        "I have chest pain.",
        "No shortness of breath.",
        "No, I have not fainted or felt faint.",
        "No, it is not getting worse quickly.",
        "This morning.",
        "Left side of my chest.",
        "About two hours.",
        "5 out of 10.",
        "Sharp.",
        "Deep breathing makes it worse.",
        "Rest helps.",
        "No other symptoms.",
        "Aspirin 81mg daily.",
        "Penicillin.",
        "No, it does not move anywhere else.",
        "No nausea.",
        "No sweating.",
        "No, it does not come on with activity.",
        "Nothing else.",
    ],
)

TEST: CHEST PAIN — full benign

INTENT SEQUENCE:
   0. ask_main_reason_for_visit
   1. ask_shortness_of_breath
   2. ask_syncope_or_presyncope
   3. ask_rapid_worsening
   4. ask_onset
   5. ask_location
   6. ask_duration
   7. ask_severity
   8. ask_character
   9. ask_aggravating_factors
  10. ask_relieving_factors
  11. ask_associated_symptoms
  12. ask_medications
  13. ask_allergies
  14. ask_radiation
  15. ask_nausea
  16. ask_diaphoresis
  17. ask_exertional_component
  18. summarize_and_check_for_anything_else
  19. summarize_and_check_for_anything_else

FINAL HPI:
{'aggravating_factors': ['deep breathing makes it worse'],
 'associated_symptoms': [],
 'character': 'Sharp',
 'context': None,
 'course': None,
 'duration': 'about two hours',
 'location': 'Left side of my chest',
 'onset': 'This morning',
 'relieving_factors': ['rest helps'],
 'severity': '5/10',
 'summary': None,
 'timing': None}

FINAL POLICY ANSWERS (non-None only):
{'aura_features': [],
 'diaphoresis': False,

In [12]:
run_policy_test(
    session_name="abdominal_full",
    primary_complaint="abdominal pain",
    label="ABDOMINAL PAIN — full",
    answers=[
        "I have stomach pain.",
        "No vomiting.",
        "No fever.",
        "No blood in my stool.",
        "No, I am not pregnant.",
        "No, I have not fainted.",
        "Last night.",
        "About twelve hours.",
        "6 out of 10.",
        "It comes and goes.",
        "It has been about the same.",
        "Around my belly button.",
        "Crampy.",
        "Eating makes it worse.",
        "Nothing really helps.",
        "Nausea.",
        "No medications.",
        "No allergies.",
        "Yes, some nausea.",
        "Yes, loose stools.",
        "No constipation.",
        "No urinary symptoms.",
        "About six hours ago.",
        "Nothing else.",
    ],
)

TEST: ABDOMINAL PAIN — full

INTENT SEQUENCE:
   0. ask_main_reason_for_visit
   1. ask_vomiting
   2. ask_fever
   3. ask_bloody_stool_or_melena
   4. ask_pregnancy_or_postpartum_context
   5. ask_syncope_or_presyncope
   6. ask_onset
   7. ask_duration
   8. ask_severity
   9. ask_timing
  10. ask_course
  11. ask_location
  12. ask_character
  13. ask_aggravating_factors
  14. ask_relieving_factors
  15. ask_associated_symptoms
  16. ask_medications
  17. ask_allergies
  18. ask_nausea
  19. ask_diarrhea
  20. ask_constipation
  21. ask_urinary_symptoms
  22. ask_last_oral_intake
  23. summarize_and_check_for_anything_else
  24. summarize_and_check_for_anything_else

FINAL HPI:
{'aggravating_factors': ['eating makes it worse'],
 'associated_symptoms': ['nausea'],
 'character': 'Crampy',
 'context': None,
 'course': 'It has been about the same',
 'duration': 'about twelve hours',
 'location': 'Around my belly button',
 'onset': 'Last night',
 'relieving_factors': [],
 'severity': '6/

In [13]:
run_policy_test(
    session_name="sob_full",
    primary_complaint="shortness of breath",
    label="SHORTNESS OF BREATH — full",
    answers=[
        "I have shortness of breath.",
        "No chest pain.",
        "No, it is not getting worse quickly.",
        "No, I have not fainted.",
        "No fever.",
        "No wheezing.",
        "This morning.",
        "About three hours.",
        "6 out of 10.",
        "It comes and goes.",
        "It has stayed about the same.",
        "Walking makes it worse.",
        "Rest helps.",
        "Cough.",
        "Albuterol as needed.",
        "No allergies.",
        "Yes, a cough.",
        "No, I can breathe fine lying flat.",
        "No leg swelling.",
        "Yes, activity makes it worse.",
        "Nothing else.",
    ],
)

TEST: SHORTNESS OF BREATH — full

INTENT SEQUENCE:
   0. ask_main_reason_for_visit
   1. ask_chest_pain
   2. ask_rapid_worsening
   3. ask_syncope_or_presyncope
   4. ask_fever
   5. ask_wheezing
   6. ask_onset
   7. ask_duration
   8. ask_severity
   9. ask_timing
  10. ask_course
  11. ask_aggravating_factors
  12. ask_relieving_factors
  13. ask_associated_symptoms
  14. ask_medications
  15. ask_allergies
  16. ask_cough
  17. ask_orthopnea
  18. ask_leg_swelling
  19. ask_exertional_component
  20. summarize_and_check_for_anything_else
  21. summarize_and_check_for_anything_else

FINAL HPI:
{'aggravating_factors': ['walking makes it worse'],
 'associated_symptoms': ['cough'],
 'character': None,
 'context': None,
 'course': 'It has stayed about the same',
 'duration': 'about three hours',
 'location': None,
 'onset': 'This morning',
 'relieving_factors': ['rest helps'],
 'severity': '6/10',
 'summary': None,
 'timing': 'It comes and goes'}

FINAL POLICY ANSWERS (non-None only):


In [14]:
run_policy_test(
    session_name="dizziness_full",
    primary_complaint="dizziness",
    label="DIZZINESS — full",
    answers=[
        "I have dizziness.",
        "No, I have not fainted or felt like I might faint.",
        "No weakness, numbness, or trouble speaking.",
        "No chest pain.",
        "No shortness of breath.",
        "No recent head trauma.",
        "This morning.",
        "About three hours.",
        "5 out of 10.",
        "It comes and goes.",
        "It has stayed about the same.",
        "Standing up makes it worse.",
        "Sitting down helps.",
        "Nausea.",
        "No medications.",
        "No allergies.",
        "No vision changes.",
        "No heart racing.",
        "Yes, it gets worse when I stand.",
        "Yes, nausea.",
        "Nothing else.",
    ],
)

TEST: DIZZINESS — full

INTENT SEQUENCE:
   0. ask_main_reason_for_visit
   1. ask_syncope_or_presyncope
   2. ask_neurologic_symptoms
   3. ask_chest_pain
   4. ask_shortness_of_breath
   5. ask_head_trauma
   6. ask_onset
   7. ask_duration
   8. ask_severity
   9. ask_timing
  10. ask_course
  11. ask_aggravating_factors
  12. ask_relieving_factors
  13. ask_associated_symptoms
  14. ask_medications
  15. ask_allergies
  16. ask_visual_changes
  17. ask_palpitations
  18. ask_positional_component
  19. ask_nausea
  20. summarize_and_check_for_anything_else
  21. summarize_and_check_for_anything_else

FINAL HPI:
{'aggravating_factors': ['standing up makes it worse'],
 'associated_symptoms': ['nausea'],
 'character': None,
 'context': None,
 'course': 'It has stayed about the same',
 'duration': 'about three hours',
 'location': None,
 'onset': 'This morning',
 'relieving_factors': ['sitting down helps'],
 'severity': '5/10',
 'summary': None,
 'timing': 'It comes and goes'}

FINAL PO

In [15]:
run_policy_test(
    session_name="sore_throat_full",
    primary_complaint="sore throat",
    label="SORE THROAT — full",
    answers=[
        "I have a sore throat.",
        "No shortness of breath.",
        "No trouble swallowing.",
        "No drooling.",
        "No fever.",
        "Yesterday evening.",
        "About sixteen hours.",
        "4 out of 10.",
        "It is constant.",
        "It has been getting slightly worse.",
        "Scratchy and raw.",
        "Swallowing makes it worse.",
        "Cold drinks help a little.",
        "Runny nose and mild cough.",
        "No medications.",
        "No allergies.",
        "Yes, a mild cough.",
        "No voice change.",
        "Yes, my partner has a cold.",
        "Nothing else.",
    ],
)

TEST: SORE THROAT — full

INTENT SEQUENCE:
   0. ask_main_reason_for_visit
   1. ask_shortness_of_breath
   2. ask_trouble_swallowing
   3. ask_drooling
   4. ask_fever
   5. ask_onset
   6. ask_duration
   7. ask_severity
   8. ask_timing
   9. ask_course
  10. ask_character
  11. ask_aggravating_factors
  12. ask_relieving_factors
  13. ask_associated_symptoms
  14. ask_medications
  15. ask_allergies
  16. ask_cough
  17. ask_voice_change
  18. ask_sick_contacts
  19. summarize_and_check_for_anything_else
  20. summarize_and_check_for_anything_else

FINAL HPI:
{'aggravating_factors': ['swallowing makes it worse'],
 'associated_symptoms': ['runny nose', 'mild cough'],
 'character': 'Scratchy and raw',
 'context': None,
 'course': 'It has been getting slightly worse',
 'duration': 'about sixteen hours',
 'location': None,
 'onset': 'Yesterday evening',
 'relieving_factors': ['cold drinks help a little'],
 'severity': '4/10',
 'summary': None,
 'timing': 'It is constant'}

FINAL POLICY

In [16]:
run_policy_test(
    session_name="back_pain_full",
    primary_complaint="back pain",
    label="BACK PAIN — full",
    answers=[
        "I have low back pain.",
        "No weakness or numbness in my legs.",
        "No bowel or bladder changes.",
        "No fever.",
        "No recent head trauma.",
        "No, it is not getting worse quickly.",
        "This morning.",
        "About six hours.",
        "6 out of 10.",
        "It is constant.",
        "It has been about the same.",
        "Lower back, right side.",
        "Dull and achy.",
        "Bending makes it worse.",
        "Lying down helps a little.",
        "No other symptoms.",
        "Ibuprofen as needed.",
        "No allergies.",
        "No, it does not go down my leg.",
        "Yes, I lifted something heavy yesterday.",
        "No numbness or tingling.",
        "Nothing else.",
    ],
)

TEST: BACK PAIN — full

INTENT SEQUENCE:
   0. ask_main_reason_for_visit
   1. ask_neurologic_symptoms
   2. ask_bowel_or_bladder_changes
   3. ask_fever
   4. ask_head_trauma
   5. ask_rapid_worsening
   6. ask_onset
   7. ask_duration
   8. ask_severity
   9. ask_timing
  10. ask_course
  11. ask_location
  12. ask_character
  13. ask_aggravating_factors
  14. ask_relieving_factors
  15. ask_associated_symptoms
  16. ask_medications
  17. ask_allergies
  18. ask_radiation
  19. ask_recent_heavy_lifting
  20. ask_numbness_or_tingling
  21. summarize_and_check_for_anything_else
  22. summarize_and_check_for_anything_else

FINAL HPI:
{'aggravating_factors': ['bending makes it worse'],
 'associated_symptoms': [],
 'character': 'Dull and achy',
 'context': None,
 'course': 'It has been about the same',
 'duration': 'about six hours',
 'location': 'Lower back, right side',
 'onset': 'This morning',
 'relieving_factors': ['lying down helps a little'],
 'severity': '6/10',
 'summary': None,


In [17]:
import intake_engine.policies.complaint_policies
print(intake_engine.policies.complaint_policies.__file__)

c:\Source\689\Capstone2026_Spring\intake_engine\policies\complaint_policies.py


In [18]:
from intake_engine.policies.complaint_policies import BACK_PAIN_POLICY
print(BACK_PAIN_POLICY["wrap_up_rule"])

{'type': 'characterization_threshold', 'require_all_critical': True, 'required_characterization_targets': ['onset', 'duration', 'severity', 'location', 'aggravating_factors', 'relieving_factors', 'associated_symptoms'], 'min_required_characterization_count': 5}


In [19]:
import intake_engine.state.rule_based_parser
print(intake_engine.state.rule_based_parser.__file__)

c:\Source\689\Capstone2026_Spring\intake_engine\state\rule_based_parser.py


In [20]:
import inspect
from intake_engine.state.rule_based_parser import build_generic_update_from_target_spec
print(inspect.getsource(build_generic_update_from_target_spec))

def build_generic_update_from_target_spec(target, answer):
    spec = TARGET_SPECS.get(target)
    if spec is None:
        return None

    state_path = spec.get("state_path")
    parse_mode = spec.get("fallback_parse_mode", "text")
    default_update_mode = spec.get("default_update_mode", "set")
    extra_set_fields = spec.get("extra_set_fields", [])
    extra_append_fields = spec.get("extra_append_fields", [])

    update = {
        "set_fields": {},
        "append_fields": {},
        "flags_to_add": [],
        "missing_clarifications_to_add": [],
    }

    update["set_fields"]["conversation_meta.last_answer_status"] = "answered"
    update["set_fields"]["conversation_meta.early_exit_reason"] = None

    if parse_mode == "yes_no":
        value = extract_yes_no_unknown(answer)

        if default_update_mode == "append":
            update["append_fields"][state_path] = [value]
        else:
            update["set_fields"][state_path] = value

        # Only append to extra fi